# 12. Blessed 30M scaling runs (baseline / Full / Block AttnRes)

Goal: run the **blessed 30M config** for all three architectures at **150M tokens each** (5 tokens/param), with **inline validation loss** and **throughput logging**. This is a real results run — **no downstream benchmark eval during training** (HellaSwag / LAMBADA / ARC run separately on saved checkpoints afterward).

- **Model**: ~30.3M params (`d_model=384`, `n_layers=6`, `n_heads=6`, `d_ff=1536`)
- **Hyperparameters**: identical to `configs/fineweb_30m_diag.yaml` (AdamW 0.9/0.95, wd 0.1, eps 1e-8, cosine + warmup, bf16, effective batch 65,536 tokens)
- **Variants** (one seed each, same seed across variants): `baseline`, `attnres` (Full), `block_attnres` (Block, `num_blocks=3`)
- **Block note**: 6 layers is too shallow for the paper's ~8-block granularity; `num_blocks=3` gives 2 transformer layers/block and is a divisibility compromise, not the target granularity.
- **Dataset**: `fineweb_edu` sample-10BT, `block_size=1024`, existing hash split
- **Logging**: train loss every 10 steps, **val loss every 100 steps**, `tokens_per_sec` + wall-clock per step, checkpoints + summaries to **Google Drive**

Flow: sync repo → mount Drive → GPU check → pre-launch report → VRAM probe → **confirmation gate** → launch 3 runs sequentially. Config: `configs/fineweb_30m_blessed.yaml`.

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/AtinChing/AttnResGPT-next-scale.git"
REPO_TARGET = Path("/content/AttnResGPT-next-scale")
DRIVE_ARTIFACT_FOLDER = "AttnResGPT-next-scale-artifacts"


def is_repo_root(path: Path) -> bool:
    return (path / "src" / "training" / "train.py").exists() and (path / "requirements.txt").exists()


def discover_repo() -> Path:
    for candidate in [REPO_TARGET, Path.cwd(), Path.cwd().parent]:
        if is_repo_root(candidate):
            return candidate.resolve()
    if not REPO_TARGET.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_TARGET)], check=True)
    return REPO_TARGET.resolve()


REPO = discover_repo()
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# W&B: enabled with mode=auto (online if WANDB_API_KEY set, else offline).
# os.environ["WANDB_API_KEY"] = "<your-key>"
print("repo:", REPO)

print("WANDB_API_KEY set =", bool(os.environ.get("WANDB_API_KEY")))

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_ARTIFACT_FOLDER
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("drive root:", DRIVE_ROOT)
print("artifacts layout:")
print("  checkpoints/", DRIVE_ROOT / "checkpoints")
print("  runs/       ", DRIVE_ROOT / "runs")
print("  logs/       ", DRIVE_ROOT / "logs")

In [ ]:
import torch

print("cuda_available =", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("device_name =", props.name)
    print("total_vram_gib =", round(props.total_memory / (1024 ** 3), 1))
    print("bf16_supported =", torch.cuda.is_bf16_supported())
else:
    print("WARNING: no CUDA device. This notebook targets a Colab A100 40GB.")

## Pre-launch report: config, 3-run plan, logging checks, cost estimate

Loads `configs/fineweb_30m_blessed.yaml`, prints the blessed hyperparameters, the three run identities, checkpoint schedule, and confirms **val loss + throughput** logging is wired (and that **no benchmark harness** runs during training).

In [ ]:
from src.utils.config import load_config
from src.models.attnres import build_model
from src.utils.logging import build_run_identity
from src.utils.runtime import count_parameters

CONFIG_PATH = "configs/fineweb_30m_blessed.yaml"
EFFECTIVE_BATCH_TOKENS = 65_536
SEED = 1337

RUN_PLAN = [
    {
        "label": "baseline",
        "architecture": "baseline",
        "overrides": ["model.architecture=baseline", "model.attnres.enabled=false"],
        "notes": "Standard PreNorm residual GPT.",
    },
    {
        "label": "full_attnres",
        "architecture": "attnres",
        "overrides": [
            "model.architecture=attnres",
            "model.attnres.enabled=true",
            "model.attnres.mode=full",
        ],
        "notes": "Full depth-attention over all prior sublayer outputs.",
    },
    {
        "label": "block_attnres",
        "architecture": "block_attnres",
        "overrides": [
            "model.architecture=block_attnres",
        ],
        "notes": (
            "Block AttnRes (num_blocks from config YAML). "
            "30M / 6 layers is too shallow for the paper's ~8-block target granularity."
        ),
    },
]

cfg = load_config(CONFIG_PATH, overrides=[f"experiment.seed={SEED}"])
eff_tokens = cfg.data.batch_size * cfg.training.grad_accum_steps * cfg.data.block_size
assert eff_tokens == EFFECTIVE_BATCH_TOKENS, f"effective batch is {eff_tokens}, expected {EFFECTIVE_BATCH_TOKENS}"

param_counts = {}
run_names = {}
for plan in RUN_PLAN:
    variant_cfg = load_config(CONFIG_PATH, overrides=[f"experiment.seed={SEED}", *plan["overrides"]])
    param_counts[plan["label"]] = count_parameters(build_model(variant_cfg.model))["total"]
    run_names[plan["label"]] = build_run_identity(variant_cfg).run_name

total_tokens = cfg.training.max_steps * eff_tokens
ckpt_steps = list(range(cfg.training.checkpoint_interval, cfg.training.max_steps + 1, cfg.training.checkpoint_interval))
if cfg.training.max_steps not in ckpt_steps:
    ckpt_steps.append(cfg.training.max_steps)

print("=== Blessed 30M scaling: pre-launch report ===")
print(f"config file   : {CONFIG_PATH}")
print(f"seed          : {SEED} (same across all 3 variants)")
print(f"model shape   : d_model={cfg.model.d_model} n_layers={cfg.model.n_layers} n_heads={cfg.model.n_heads} d_ff={cfg.model.d_ff}")
print(f"param counts  : baseline={param_counts['baseline']:,}  full={param_counts['full_attnres']:,}  block={param_counts['block_attnres']:,}")
print(f"optimizer     : AdamW betas=({cfg.training.beta1},{cfg.training.beta2}) eps={cfg.training.adam_eps} wd={cfg.training.weight_decay}")
print(f"schedule      : cosine + warmup | peak={cfg.training.learning_rate} min={cfg.training.min_lr} warmup={cfg.training.warmup_steps}")
print(f"precision     : {'bf16' if cfg.training.mixed_precision else 'fp32'} ({cfg.training.amp_dtype})")
print(f"effective batch: {cfg.data.batch_size} micro x {cfg.training.grad_accum_steps} accum x {cfg.data.block_size} ctx = {eff_tokens:,} tokens/step")
print(f"horizon/run   : max_steps={cfg.training.max_steps} -> {total_tokens:,} tokens ({total_tokens/1e6:.1f}M)")
print(f"total budget  : {3 * total_tokens:,} tokens across 3 runs ({3 * total_tokens/1e6:.1f}M)")
print()
print("=== logging (train.py) ===")
print(f"train loss    : every {cfg.training.log_interval} steps -> train_metrics.jsonl + W&B")
print(f"val loss      : every {cfg.training.eval_interval} steps -> val_metrics.jsonl + W&B (ENABLED)")
print("throughput    : tokens_per_sec, step_time_sec, elapsed_time_sec logged each train step")
print("benchmarks    : NONE during training (no lm-eval / ARC / LAMBADA / HellaSwag)")
assert cfg.training.eval_interval > 0, "eval_interval must be > 0 for val loss curves"
print()
print("=== 3-run plan ===")
for plan in RUN_PLAN:
    print(f"  {plan['label']:14s} arch={plan['architecture']:<13s} run_name={run_names[plan['label']]}")
    print(f"                 {plan['notes']}")
print()
print(f"checkpoints   : every {cfg.training.checkpoint_interval} steps (~{cfg.training.checkpoint_interval * eff_tokens / 1e6:.1f}M tokens), keep_last_k={cfg.logging.keep_last_k_checkpoints}")
for step in ckpt_steps:
    print(f"    step {step:>5}  ~{step * eff_tokens / 1e6:6.1f}M tokens")
print(f"  checkpoints per run: {len(ckpt_steps)} (written to Drive)")
print()
# Cost estimate: 3 runs x 150M tokens each on ~30M params
param_ref = param_counts["baseline"]
train_flops = 3 * 6 * param_ref * total_tokens
A100_BF16_PEAK_FLOPS = 312e12
for mfu, label in [(0.30, "conservative"), (0.45, "optimistic")]:
    eff_flops = mfu * A100_BF16_PEAK_FLOPS
    hours = train_flops / eff_flops / 3600
    print(f"est wall-clock all 3 runs ({label}, MFU {mfu:.0%}): {hours:.2f} h  (~{3 * total_tokens / (hours * 3600):,.0f} tok/s effective)")
print("est Colab A100 cost: ~12 CU/h => roughly 3-8 CU total for all 3 runs (approximate).")

## VRAM probe: pick the largest micro-batch that fits (effective batch stays 65,536 tokens)

30M models are smaller than the 200M probe; probes micro-batches 64 → 32 → 16 → 8 and sets `grad_accum` to hold 64 sequences per optimizer step.

In [ ]:
from torch.optim import AdamW
from src.metrics.norms import language_model_loss
from src.utils.runtime import amp_dtype_from_string

EFFECTIVE_BATCH_SEQUENCES = EFFECTIVE_BATCH_TOKENS // cfg.data.block_size  # 64
CANDIDATE_MICRO_BATCHES = [64, 32, 16, 8]
VRAM_HEADROOM = 0.90

MICRO_BATCH = cfg.data.batch_size
GRAD_ACCUM = cfg.training.grad_accum_steps


def _peak_mib_for_micro_batch(micro_batch: int, architecture: str = "baseline", steps: int = 3) -> float:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    variant_cfg = load_config(CONFIG_PATH, overrides=[f"model.architecture={architecture}"])
    model = build_model(variant_cfg.model).to("cuda")
    opt = AdamW(
        model.parameters(),
        lr=cfg.training.learning_rate,
        betas=(cfg.training.beta1, cfg.training.beta2),
        eps=cfg.training.adam_eps,
        weight_decay=cfg.training.weight_decay,
    )
    amp_dtype = amp_dtype_from_string(cfg.training.amp_dtype)
    model.train()
    for _ in range(steps):
        x = torch.randint(0, cfg.model.vocab_size, (micro_batch, cfg.data.block_size), device="cuda")
        y = torch.randint(0, cfg.model.vocab_size, (micro_batch, cfg.data.block_size), device="cuda")
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=cfg.training.mixed_precision):
            logits, _ = model(x, return_aux=False)
            loss = language_model_loss(logits, y)
        loss.backward()
        opt.step()
    peak = torch.cuda.max_memory_allocated() / (1024 ** 2)
    del model, opt
    torch.cuda.empty_cache()
    return peak


if torch.cuda.is_available():
    total_mib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
    budget = VRAM_HEADROOM * total_mib
    print(f"VRAM budget: {budget:,.0f} / {total_mib:,.0f} MiB @ {VRAM_HEADROOM:.0%}")
    chosen = None
    # Probe with block_attnres (largest variant) to be safe.
    for mb in sorted(CANDIDATE_MICRO_BATCHES, reverse=True):
        if EFFECTIVE_BATCH_SEQUENCES % mb != 0:
            continue
        try:
            peak = _peak_mib_for_micro_batch(mb, architecture="block_attnres")
        except torch.cuda.OutOfMemoryError:
            print(f"micro_batch={mb:>3}: OOM")
            torch.cuda.empty_cache()
            continue
        fits = peak <= budget
        print(f"micro_batch={mb:>3}: peak {peak:8,.0f} MiB [{'OK' if fits else 'over budget'}]")
        if fits and chosen is None:
            chosen = mb
    if chosen is not None:
        MICRO_BATCH = chosen
        GRAD_ACCUM = EFFECTIVE_BATCH_SEQUENCES // chosen
    print(f"\nchosen micro_batch={MICRO_BATCH}, grad_accum={GRAD_ACCUM} "
          f"-> {MICRO_BATCH * GRAD_ACCUM} seqs = {MICRO_BATCH * GRAD_ACCUM * cfg.data.block_size:,} tokens/step")
else:
    print(f"No CUDA; using config defaults micro_batch={MICRO_BATCH}, grad_accum={GRAD_ACCUM}")

## Confirmation gate (must be True to launch)

Review the config, 3-run plan, cost estimate, and chosen micro-batch above. The next cell launches **baseline → Full AttnRes → Block AttnRes** sequentially, writing checkpoints and val-loss logs directly to Drive. Set `CONFIRM_LAUNCH = True` only when ready.

In [ ]:
CONFIRM_LAUNCH = False

SHARED_OVERRIDES = [
    f"logging.output_root={DRIVE_ROOT}",
    f"experiment.seed={SEED}",
    f"data.batch_size={MICRO_BATCH}",
    f"data.eval_batch_size={MICRO_BATCH}",
    f"training.grad_accum_steps={GRAD_ACCUM}",
    "logging.wandb.group=blessed_30m_scaling",
]

print("final launch plan:")
print("  config        :", CONFIG_PATH)
print("  output_root   :", DRIVE_ROOT)
print("  shared overrides:", SHARED_OVERRIDES)
print("  runs:")
for plan in RUN_PLAN:
    print(f"    - {plan['label']}: {run_names[plan['label']]}")
print("  CONFIRM_LAUNCH =", CONFIRM_LAUNCH)
print("\nSet CONFIRM_LAUNCH = True, then run the next cell to start all 3 runs (~150M tokens each).")

In [ ]:
if not CONFIRM_LAUNCH:
    raise RuntimeError("Set CONFIRM_LAUNCH = True in the previous cell after reviewing the plan.")

from src.training.train import train_from_config

RUN_SUMMARIES = []
suite_start = time.time()

for plan in RUN_PLAN:
    label = plan["label"]
    print("\n" + "=" * 72)
    print(f"LAUNCHING: {label}  ({plan['architecture']})")
    print(plan["notes"])
    print("=" * 72)

    overrides = SHARED_OVERRIDES + plan["overrides"] + [f"logging.wandb.run_name={run_names[label]}"]
    launch_cfg = load_config(CONFIG_PATH, overrides=overrides)

    assert launch_cfg.training.eval_interval > 0, "val loss must be enabled"
    assert (
        launch_cfg.data.batch_size
        * launch_cfg.training.grad_accum_steps
        * launch_cfg.data.block_size
        == EFFECTIVE_BATCH_TOKENS
    ), "effective batch drifted from 65,536 tokens"

    run_start = time.time()
    summary = train_from_config(launch_cfg)
    run_hours = (time.time() - run_start) / 3600

    RUN_SUMMARIES.append(
        {
            "label": label,
            "architecture": plan["architecture"],
            "run_name": summary["run_name"],
            "tokens_seen": summary.get("tokens_seen"),
            "elapsed_time_sec": summary.get("elapsed_time_sec"),
            "wall_clock_hours": run_hours,
            "best_val_loss": summary.get("best_val_loss"),
            "val_loss_final": summary.get("val_loss"),
            "checkpoint_path": summary.get("checkpoint_path"),
            "wandb_url": summary.get("wandb_url"),
        }
    )

    print(f"\nFinished {label} in {run_hours:.2f} h")
    print("  run_name       :", summary["run_name"])
    print("  tokens_seen    :", summary.get("tokens_seen"))
    print("  best_val_loss  :", summary.get("best_val_loss"))
    print("  final val_loss :", summary.get("val_loss"))
    print("  checkpoint     :", summary.get("checkpoint_path"))
    print("  wandb          :", summary.get("wandb_url"))

suite_hours = (time.time() - suite_start) / 3600
print("\n" + "=" * 72)
print(f"ALL 3 RUNS COMPLETE in {suite_hours:.2f} h wall-clock")
for row in RUN_SUMMARIES:
    tok = row.get("tokens_seen") or 0
    elapsed = row.get("elapsed_time_sec") or 1
    tps = tok / max(elapsed, 1e-6)
    print(
        f"  {row['label']:14s}  val_loss={row.get('val_loss_final')}  "
        f"best_val={row.get('best_val_loss')}  throughput={tps:,.0f} tok/s  "
        f"wall={row['wall_clock_hours']:.2f}h"
    )

_TRAINING_COMPLETED = True

## Resume a single variant after a disconnect

Re-run cells 1–8 on a fresh runtime, set `RESUME_LABEL` to the variant that was interrupted (`baseline`, `full_attnres`, or `block_attnres`), set `CONFIRM_RESUME = True`, then run the cell below. It resumes from the **latest checkpoint on Drive** for that run. Do **not** also run the fresh-launch cell in the same pass.

In [ ]:
CONFIRM_RESUME = False
RESUME_LABEL = "baseline"  # baseline | full_attnres | block_attnres

if not CONFIRM_RESUME:
    print("Resume skipped. Set CONFIRM_RESUME=True and RESUME_LABEL after a disconnect.")
else:
    from src.training.train import train_from_config

    plan = next(item for item in RUN_PLAN if item["label"] == RESUME_LABEL)
    run_name = run_names[RESUME_LABEL]
    resume_dir = DRIVE_ROOT / "checkpoints" / run_name
    ckpts = sorted(resume_dir.glob("step_*.pt"))
    if not ckpts:
        raise FileNotFoundError(f"No checkpoints found on Drive at {resume_dir}")
    latest_ckpt = ckpts[-1]
    print(f"resuming {RESUME_LABEL} from:", latest_ckpt)

    resume_overrides = SHARED_OVERRIDES + plan["overrides"] + [
        f"logging.wandb.run_name={run_name}",
        f"training.resume_from={latest_ckpt}",
    ]
    resume_cfg = load_config(CONFIG_PATH, overrides=resume_overrides)
    assert resume_cfg.training.eval_interval > 0

    resume_start = time.time()
    summary = train_from_config(resume_cfg)
    print(f"\nResumed {RESUME_LABEL} finished in {(time.time() - resume_start)/3600:.2f} h")
    print("checkpoints on Drive:", resume_dir)
    print("summary:", {k: summary.get(k) for k in ["run_name", "tokens_seen", "best_val_loss", "val_loss", "checkpoint_path"]})
    _TRAINING_COMPLETED = True

## End the Colab session on completion

Unassigns the runtime after all runs finish so idle compute units aren't burned.

In [ ]:
UNASSIGN_ON_COMPLETE = True

training_completed = globals().get("_TRAINING_COMPLETED", False)

if UNASSIGN_ON_COMPLETE and training_completed:
    print("Training complete - ending Colab session to free the A100.")
    from google.colab import runtime
    runtime.unassign()
else:
    print(f"Not unassigning (UNASSIGN_ON_COMPLETE={UNASSIGN_ON_COMPLETE}, training_completed={training_completed}).")